# Newton's method for logistic regression

A nursery records whether each seed tray germinated, with the soil temperature and the
moisture, both already standardized. You will write the loss of a logistic regression with
a ridge penalty that leaves the intercept alone, its gradient and its Hessian, then fit it
with Newton's method and see the penalty give a finite answer on data that a line separates.

**How to work through it.** Each part asks for one function. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your function is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first.

In [ ]:
import warnings
import numpy as np

# 400 trays: column 0 is ones, then soil temperature and moisture (standardized).
rng = np.random.default_rng(7)
n = 400
features = rng.normal(size=(n, 2))
z_true = -1.0 + 1.5 * features[:, 0] - 0.8 * features[:, 1]
y = (rng.random(n) < 1 / (1 + np.exp(-z_true))).astype(float)
X = np.column_stack([np.ones(n), features])

# Six trays that one threshold on temperature separates perfectly.
X_sep = np.column_stack([np.ones(6), [-2.0, -1.5, -0.5, 0.5, 1.2, 2.0]])
y_sep = np.array([0.0, 0.0, 0.0, 1.0, 1.0, 1.0])
print(X.shape, f"{int(y.sum())} of {n} trays germinated")

## Part 1 (coding task): The loss, its gradient and its Hessian

Write `loss_grad_hess(w, X, y, lam=0.0)`. For row $i$ let $z_i = \mathbf{x}_i^\top\mathbf{w}$
and $p_i = \sigma(z_i)$. The loss is the mean log loss plus a ridge penalty on every weight
except the intercept $w_0$:

$$
L(\mathbf{w}) = \frac1n \sum_{i=1}^n \Big[\log\big(1 + e^{z_i}\big) - y_i z_i\Big] + \frac{\lambda}{2}\sum_{j \ge 1} w_j^2 .
$$

The bracket equals $-y_i \log p_i - (1 - y_i)\log(1 - p_i)$, written so that it never
takes the log of 0. Return `(loss, grad, hess)`, where

$$
\nabla L = \frac1n X^\top(\mathbf{p} - \mathbf{y}) + \lambda\,\mathbf{w}_{\text{pen}}, \qquad
\nabla^2 L = \frac1n X^\top S X + \lambda\,\operatorname{diag}(0, 1, \dots, 1),
$$

$\mathbf{w}_{\text{pen}}$ is $\mathbf{w}$ with its first entry set to 0, and $S$ is the
diagonal matrix whose entry $i$ is $p_i(1 - p_i)$. Two things to get right: `np.exp` of a large
$z$ overflows, and `np.logaddexp(0, z)` computes $\log(1 + e^{z})$ without overflow, as
`0.5 * (1 + np.tanh(z / 2))` computes $\sigma(z)$. And build $X^\top S X$ by scaling the rows
of $X$, without forming the $n \times n$ matrix $S$.

In [ ]:
def loss_grad_hess(w, X, y, lam=0.0):
    """Mean log loss plus (lam / 2) * sum(w[1:] ** 2), and its gradient and Hessian."""
    # TODO
    raise NotImplementedError

In [ ]:
L0, g0, H0 = loss_grad_hess(np.zeros(3), X, y)
assert np.isclose(L0, np.log(2)), f"at w = 0 every p is 0.5, so the mean loss is log 2 = 0.693, not {L0:.4f}: use the mean, not the sum"
assert np.allclose(g0, X.T @ (0.5 - y) / n), "at w = 0 the gradient is X^T (0.5 - y) / n"

w_try = np.array([0.4, -0.7, 1.1])
lam = 0.3
L1, g1, H1 = loss_grad_hess(w_try, X, y, lam)
L_plain = loss_grad_hess(w_try, X, y, 0.0)[0]
assert np.isclose(L1 - L_plain, lam / 2 * (0.7 ** 2 + 1.1 ** 2)), "the penalty should be (lam / 2) * sum(w[1:] ** 2): check the factor 1/2, and leave the intercept w[0] out of it"

def central_difference(which, h=1e-6):
    """Nudge each weight up and down by h and difference output `which` (0 loss, 1 gradient)."""
    cols = []
    for j in range(3):
        step = np.zeros(3)
        step[j] = h
        up, down = loss_grad_hess(w_try + step, X, y, lam)[which], loss_grad_hess(w_try - step, X, y, lam)[which]
        cols.append((up - down) / (2 * h))
    return np.array(cols)


fd_grad, fd_hess = central_difference(0), central_difference(1)
assert np.allclose(g1, fd_grad, atol=1e-6), f"the gradient does not match finite differences of the loss: {g1} against {fd_grad}"
assert np.allclose(H1, fd_hess, atol=1e-5), "the Hessian does not match finite differences of the gradient"
assert H1.shape == (3, 3), f"the Hessian should be 3 x 3, not {H1.shape}"

with warnings.catch_warnings():
    warnings.simplefilter("error")                  # an overflow warning fails the check
    try:
        L_big = loss_grad_hess(np.array([0.0, 800.0, 0.0]), X, y)[0]
    except (RuntimeWarning, FloatingPointError) as err:
        raise AssertionError(f"the loss overflowed for huge weights ({err}): use np.logaddexp(0, z) and the tanh form of the sigmoid") from None
assert np.isfinite(L_big), "the loss must stay finite when the weights are huge"
print("Part 1 passed")

## Part 2 (coding task): Newton's method

Write `newton(X, y, lam=0.0, tol=1e-10, max_steps=50)`. Start from $\mathbf{w} = \mathbf{0}$ and
repeat the full Newton step $\mathbf{w} \leftarrow \mathbf{w} - (\nabla^2 L)^{-1}\nabla L$, using
`np.linalg.solve` rather than an inverse, until the gradient's norm is below `tol` or
`max_steps` steps have been taken. Return `(w, steps)`, the final weights and the number of
steps taken.

The check fits the trays, then the separable data with a small penalty, then a penalty so
large that every weight but the intercept is pushed to zero.

In [ ]:
def newton(X, y, lam=0.0, tol=1e-10, max_steps=50):
    w = np.zeros(X.shape[1])
    # TODO: Newton steps until the gradient's norm is below tol, at most max_steps of them.
    raise NotImplementedError

In [ ]:
w_fit, steps = newton(X, y)
print("trays:", np.round(w_fit, 3), "after", steps, "steps")
assert np.linalg.norm(loss_grad_hess(w_fit, X, y)[1]) < 1e-8, "the gradient is not zero at the returned weights"
assert 3 <= steps <= 10, f"Newton's method should take a handful of full steps here, not {steps}"

w_pen, steps_pen = newton(X_sep, y_sep, lam=0.1)
print("separable, lam = 0.1:", np.round(w_pen, 3), "after", steps_pen, "steps")
assert steps_pen < 50 and np.linalg.norm(loss_grad_hess(w_pen, X_sep, y_sep, 0.1)[1]) < 1e-8, "with lam = 0.1 the separable fit should converge"
assert np.abs(w_pen).max() < 10, "the penalty should keep the weights finite and moderate"

Xc = X.copy()
Xc[:, 1:] -= Xc[:, 1:].mean(axis=0)                 # centered features
w_huge, _ = newton(Xc, y, lam=1e6)
base = np.log(y.mean() / (1 - y.mean()))
assert np.abs(w_huge[1:]).max() < 1e-3, "a huge lam should push the feature weights to zero"
assert abs(w_huge[0] - base) < 1e-4, f"with the features at zero weight the intercept should be the log-odds of the germination rate, {base:.4f}, not {w_huge[0]:.4f}: is it being penalized?"
print("Part 2 passed")

## Solution

### Part 1: The loss, its gradient and its Hessian

In [ ]:
def loss_grad_hess(w, X, y, lam=0.0):
    """Mean log loss plus (lam / 2) * sum(w[1:] ** 2), and its gradient and Hessian."""
    n = len(y)
    z = X @ w
    p = 0.5 * (1 + np.tanh(z / 2))                  # sigmoid without overflow
    w_pen = w.copy()
    w_pen[0] = 0.0                                  # the intercept is not penalized
    penalty = 0.5 * lam * (w_pen @ w_pen)
    loss = np.logaddexp(0, z).mean() - (y * z).mean() + penalty
    grad = X.T @ (p - y) / n + lam * w_pen
    pen = np.full(len(w), lam)
    pen[0] = 0.0
    hess = X.T @ (X * (p * (1 - p))[:, None]) / n + np.diag(pen)
    return loss, grad, hess

**Why it works.** At $w = 0$ every $z_i$ is 0, so each term is $\log 2$ and their mean is $\log 2$, while
a summed loss would give $400 \log 2$. `np.logaddexp(0, z)` evaluates $\log(1 + e^{z})$ as
$\max(0, z) + \log(1 + e^{-\lvert z \rvert})$, which never exponentiates a positive number, and the `tanh` form of the sigmoid never exponentiates
anything large, so a weight of 800 gives a large but finite loss instead of `inf` or `nan`.
Zeroing the first entry of a copy of $\mathbf{w}$ leaves the intercept out of the penalty, of
its gradient and of the Hessian's diagonal in one place. `X * (p * (1 - p))[:, None]` scales
row $i$ by $S_{ii}$, which is $SX$ without the $n \times n$ diagonal matrix.

### Part 2: Newton's method

In [ ]:
def newton(X, y, lam=0.0, tol=1e-10, max_steps=50):
    w = np.zeros(X.shape[1])
    for steps in range(max_steps + 1):
        _, grad, hess = loss_grad_hess(w, X, y, lam)
        if np.linalg.norm(grad) < tol or steps == max_steps:
            return w, steps
        w = w - np.linalg.solve(hess, grad)

**Why it works.** The loss is convex and near its minimum it is close to a quadratic, which one Newton
step minimizes exactly, so the gradient's norm falls roughly by squaring each step once it
is small. On the trays that takes 6 steps where gradient descent needs thousands. On the
separable trays the unpenalized loss has no minimum, so Newton's steps would grow the
weights without end, and the penalty adds $\lambda$ to the Hessian's diagonal, which gives
the loss a minimum and keeps the solve well posed. With a huge $\lambda$ the feature weights
go to zero, and the intercept's own gradient, the mean of $p_i - y_i$, can only vanish when
$\sigma(w_0)$ equals the germination rate, which is why an unpenalized intercept ends at the
log-odds of the base rate.